# Reinforcement Learning — Học tăng cường

## 1. Đặt vấn đề

Đến giờ, các bài lab đều thuộc loại **học có giám sát**: cho input + label, học hàm $f(x) \to y$. Nhưng nhiều bài toán không có "đáp án đúng" sẵn:
- Đi tới đâu trong bàn cờ vây để thắng?
- Robot nên đẩy chân nào để tiến về đích?
- Xe tự lái nên rẽ trái hay đi thẳng?

Ở những bài này, ta chỉ biết: "sau hàng loạt hành động, nếu đạt mục tiêu thì có *thưởng* (reward), nếu thất bại thì *phạt*". Đây là lúc **Reinforcement Learning** (RL) lên ngôi.

### Cơ chế cơ bản

Một agent (tác tử) tương tác với một môi trường (environment) qua các bước thời gian:

1. Tại bước $t$, môi trường ở **state** $s_t$.
2. Agent chọn **action** $a_t$ theo chính sách $\pi(a | s)$.
3. Môi trường chuyển sang state mới $s_{t+1}$ và trả **reward** $r_t$.
4. Lặp lại cho đến khi kết thúc episode.

Mục tiêu: tìm chính sách $\pi$ tối đa hoá **return tích luỹ**:
$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$$

với $\gamma \in [0, 1)$ là **discount factor** — quan tâm đến reward gần hiện tại hơn reward xa tương lai.

## 2. Các khái niệm trung tâm

### Value function
$V^\pi(s) = \mathbb{E}_\pi[G_t \mid s_t = s]$ — kỳ vọng tổng reward nếu bắt đầu ở state $s$ và đi theo chính sách $\pi$.

### Q-function (action-value)
$Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid s_t = s, a_t = a]$ — kỳ vọng tổng reward nếu ở state $s$ chọn action $a$ rồi mới đi theo $\pi$.

### Phương trình Bellman
$$Q^\pi(s, a) = r(s, a) + \gamma \sum_{s'} P(s' | s, a) \sum_{a'} \pi(a' | s') Q^\pi(s', a')$$

Đây là phương trình hồi quy: giá trị Q hiện tại = reward ngay + discount × giá trị Q tương lai.

Ta sẽ tập trung vào **Q-learning** — một thuật toán RL không cần biết $P(s'|s,a)$, chỉ cần tương tác với môi trường.

### Sơ đồ vòng lặp Agent ↔ Environment


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

# Agent box
agent = FancyBboxPatch((1, 3.5), 2.5, 1.5, boxstyle='round,pad=0.1',
                       facecolor='#cce5ff', edgecolor='black', linewidth=1.5)
ax.add_patch(agent)
ax.text(2.25, 4.25, 'Agent', ha='center', va='center', fontsize=14, fontweight='bold')

# Environment box
env = FancyBboxPatch((6, 3.5), 2.5, 1.5, boxstyle='round,pad=0.1',
                     facecolor='#fff3cd', edgecolor='black', linewidth=1.5)
ax.add_patch(env)
ax.text(7.25, 4.25, 'Environment', ha='center', va='center', fontsize=14, fontweight='bold')

# Action arrow (Agent -> Env)
ax.add_patch(FancyArrowPatch((3.5, 4.6), (6, 4.6), arrowstyle='->',
                             mutation_scale=20, color='black', linewidth=2))
ax.text(4.75, 4.85, 'action $a_t$', ha='center', fontsize=12, color='darkred')

# State + Reward arrow (Env -> Agent)
ax.add_patch(FancyArrowPatch((6, 3.9), (3.5, 3.9), arrowstyle='->',
                             mutation_scale=20, color='black', linewidth=2))
ax.text(4.75, 3.55, 'state $s_{t+1}$,  reward $r_t$', ha='center', fontsize=12, color='darkgreen')

# Cycle indicator
ax.text(4.75, 2.5, '↻  Lặp lại đến hết episode', ha='center', fontsize=11, style='italic')

# Goal box at bottom
ax.text(5, 1.2, 'Mục tiêu: chọn $\\pi(a|s)$ tối đa hoá $G_t = \\sum_{k=0}^{\\infty}\\gamma^k r_{t+k}$',
        ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgray'))

ax.set_title('Reinforcement Learning loop: Agent ↔ Environment', fontsize=14)
plt.tight_layout(); plt.show()


## 3. Q-Learning

Ý tưởng: học một bảng $Q[s, a]$ chứa giá trị Q ước lượng. Cập nhật bằng công thức:
$$
Q[s, a] \leftarrow Q[s, a] + \alpha \big(r + \gamma \max_{a'} Q[s', a'] - Q[s, a]\big)
$$

Trong đó:
- $\alpha$ là learning rate.
- $r + \gamma \max_{a'} Q[s', a']$ là **target** — giá trị Q nên bằng nếu agent chọn tối ưu ở bước kế.
- Hiệu giữa target và Q hiện tại gọi là **TD error** (temporal difference).

### Khám phá vs khai thác (Exploration vs Exploitation)

Nếu agent luôn chọn action có Q lớn nhất → "khai thác" (exploitation), không khám phá hành động mới.
Nếu chọn ngẫu nhiên → "khám phá" (exploration), không tận dụng kiến thức đã học.

Cân bằng: **ε-greedy** — với xác suất $\epsilon$ chọn ngẫu nhiên, ngược lại chọn action có Q max. Thường $\epsilon$ giảm dần theo thời gian (lúc đầu khám phá nhiều, sau ít dần).

### Trực quan: ε-greedy và discount factor

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Minh hoạ ε-greedy — vì sao cần exploration.
np.random.seed(0)
n_arms = 5
true_means = np.array([0.1, 0.3, 0.5, 0.7, 0.4])  # 5 cánh tay
n_steps = 200

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, eps, color in [('ε=0 (greedy)', 0.0, 'red'),
                          ('ε=0.1', 0.1, 'blue'),
                          ('ε=1.0 (random)', 1.0, 'gray')]:
    np.random.seed(0)
    Q = np.zeros(n_arms); n = np.zeros(n_arms); rewards = []
    for t in range(n_steps):
        if np.random.rand() < eps:
            a = np.random.randint(n_arms)
        else:
            a = int(np.argmax(Q))
        r = np.random.randn() * 0.1 + true_means[a]
        n[a] += 1
        Q[a] += (r - Q[a]) / n[a]
        rewards.append(r)
    cum = np.cumsum(rewards) / np.arange(1, n_steps + 1)
    axes[0].plot(cum, label=label, color=color)

axes[0].axhline(true_means.max(), color='black', linestyle='--', alpha=0.5, label='Optimal')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Reward trung bình')
axes[0].set_title('ε-greedy: cân bằng Exploration vs Exploitation')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Discount factor
gammas = [0.5, 0.9, 0.99]
ts = np.arange(50)
for g in gammas:
    axes[1].plot(ts, g**ts, label=f'γ={g}')
axes[1].set_xlabel('Bước thời gian k'); axes[1].set_ylabel('$\\gamma^k$')
axes[1].set_title('Discount factor: reward càng xa càng được \"giảm giá\"')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


# THỰC HÀNH 1: Q-Learning trên FrozenLake

**FrozenLake** là môi trường đơn giản trong gym:
- Lưới 4×4. Agent bắt đầu ở góc trái trên, mục tiêu là góc phải dưới.
- Có một số ô "hố" — rơi vào là kết thúc episode, reward 0.
- Đến đích → reward 1.
- Hai biến thể: `is_slippery=True` (đi trượt — khó), `False` (đi đúng hướng — dễ). Ta dùng phiên bản dễ trước.

Cài thư viện trước (chỉ chạy 1 lần):
```bash
pip install gymnasium
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

try:
    import gymnasium as gym
except ImportError:
    print('Chưa cài gymnasium — chạy: pip install gymnasium')
    raise

np.random.seed(42)
random.seed(42)

env = gym.make('FrozenLake-v1', is_slippery=False)
n_states  = env.observation_space.n      # 16
n_actions = env.action_space.n           # 4 (trái, dưới, phải, lên)
print(f'Số state: {n_states}, số action: {n_actions}')

# Bản đồ FrozenLake mặc định:
# S F F F
# F H F H
# F F F H
# H F F G
# S = start, F = frozen (đi được), H = hole, G = goal.

In [ ]:
Q = np.zeros((n_states, n_actions))

# Hyperparams
alpha    = 0.8       # learning rate
gamma    = 0.95      # discount factor
epsilon  = 1.0       # exploration rate (giảm dần)
eps_decay = 0.995
eps_min  = 0.01
n_episodes = 2000

rewards_history = []

for ep in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        # ε-greedy: chọn ngẫu nhiên với xác suất ε.
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Cập nhật Q-table.
        Q[state, action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])

        state = next_state
        total_reward += reward

    epsilon = max(eps_min, epsilon * eps_decay)
    rewards_history.append(total_reward)

    if (ep + 1) % 200 == 0:
        avg = np.mean(rewards_history[-100:])
        print(f'Episode {ep+1:4d}  avg reward last 100: {avg:.2f}  ε = {epsilon:.3f}')

In [ ]:
# Vẽ tỷ lệ thành công theo episode (rolling window 100).
rewards_arr = np.array(rewards_history)
rolling = np.convolve(rewards_arr, np.ones(100)/100, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(rolling)
plt.xlabel('Episode'); plt.ylabel('Tỷ lệ thắng (rolling 100)')
plt.title('Q-Learning trên FrozenLake (deterministic)')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# Kiểm tra chính sách đã học bằng cách chạy 100 episode greedy.
wins = 0
for _ in range(100):
    state, _ = env.reset()
    done = False
    while not done:
        action = int(np.argmax(Q[state]))
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        if reward > 0: wins += 1
print(f'Tỷ lệ thắng greedy: {wins}/100')

# In bảng Q-table cho 4 ô đầu tiên
print('\nQ-table (4 hàng đầu — state 0,1,2,3):')
print('Action: [trái, dưới, phải, lên]')
for s in range(4):
    print(f'State {s}: {Q[s].round(3)}')

### Trực quan chính sách

Vẽ mũi tên trên bản đồ 4×4 thể hiện action mà agent chọn ở mỗi state.

In [ ]:
arrows = ['←', '↓', '→', '↑']
lake_map = ['S', 'F', 'F', 'F',
            'F', 'H', 'F', 'H',
            'F', 'F', 'F', 'H',
            'H', 'F', 'F', 'G']

fig, ax = plt.subplots(figsize=(5, 5))
for i in range(4):
    for j in range(4):
        s = i * 4 + j
        cell = lake_map[s]
        if cell in ('S', 'G'):
            color = 'lightgreen' if cell == 'G' else 'lightyellow'
        elif cell == 'H':
            color = 'lightcoral'
        else:
            color = 'lightblue'
        ax.add_patch(plt.Rectangle((j, 3-i), 1, 1, facecolor=color, edgecolor='black'))
        if cell == 'F' or cell == 'S':
            best_a = int(np.argmax(Q[s]))
            ax.text(j + 0.5, 3-i + 0.5, arrows[best_a], ha='center', va='center', fontsize=20)
        else:
            ax.text(j + 0.5, 3-i + 0.5, cell, ha='center', va='center', fontsize=18, fontweight='bold')
ax.set_xlim(0, 4); ax.set_ylim(0, 4); ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Chính sách đã học (mũi tên = action greedy)')
plt.show()

# THỰC HÀNH 2: Deep Q-Network (DQN) trên CartPole

Q-table chỉ làm được khi state space hữu hạn và nhỏ. Với môi trường có state liên tục (CartPole: vị trí, vận tốc, góc, vận tốc góc), ta dùng một mạng nơ-ron để xấp xỉ Q-function. Đây là **DQN** — bài báo nổi tiếng của DeepMind 2013.

**CartPole**: cây sào dựng trên xe trượt. Agent đẩy xe trái/phải để giữ sào không ngã. Mỗi bước sào còn đứng → +1 reward, ngã → kết thúc.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]   # 4
action_dim = env.action_space.n              # 2
print('state_dim:', state_dim, ' action_dim:', action_dim)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
    def forward(self, s):
        return self.net(s)

policy_net = QNetwork(state_dim, action_dim).to(device)
# Target network: bản sao tách riêng, cập nhật chậm để training ổn định.
target_net = QNetwork(state_dim, action_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Replay buffer: lưu các transition (s, a, r, s', done) để sample mini-batch.
# Phá vỡ tương quan thời gian — quan trọng để training ổn định.
replay = deque(maxlen=10000)
batch_size = 64
gamma = 0.99
epsilon = 1.0
eps_decay = 0.995
eps_min = 0.05
target_update_freq = 10   # update target_net mỗi 10 episode

In [ ]:
def select_action(state, eps):
    if random.random() < eps:
        return env.action_space.sample()
    with torch.no_grad():
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        return int(policy_net(s).argmax(dim=1).item())

def train_step():
    if len(replay) < batch_size:
        return
    batch = random.sample(replay, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    states      = torch.FloatTensor(np.array(states)).to(device)
    actions     = torch.LongTensor(actions).unsqueeze(1).to(device)
    rewards     = torch.FloatTensor(rewards).to(device)
    next_states = torch.FloatTensor(np.array(next_states)).to(device)
    dones       = torch.FloatTensor(dones).to(device)

    # Q-value hiện tại với action đã chọn.
    q_pred = policy_net(states).gather(1, actions).squeeze(1)
    # Target: r + γ max_a' Q_target(s', a'), với done=1 thì không cộng phần future.
    with torch.no_grad():
        q_next = target_net(next_states).max(1)[0]
        q_target = rewards + gamma * q_next * (1 - dones)

    loss = criterion(q_pred, q_target)
    optimizer.zero_grad(); loss.backward()
    # Clip gradient để tránh exploding.
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 10.0)
    optimizer.step()
    return loss.item()

In [ ]:
n_episodes = 300
ep_rewards = []

for ep in range(n_episodes):
    state, _ = env.reset(seed=ep)
    total = 0
    done = False
    while not done:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay.append((state, action, reward, next_state, float(done)))
        train_step()
        state = next_state
        total += reward

    epsilon = max(eps_min, epsilon * eps_decay)
    ep_rewards.append(total)

    if ep % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if (ep + 1) % 30 == 0:
        avg = np.mean(ep_rewards[-30:])
        print(f'Ep {ep+1:3d}  avg reward last 30: {avg:6.2f}  ε = {epsilon:.3f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ep_rewards, alpha=0.4, label='Reward mỗi episode')
if len(ep_rewards) >= 30:
    rolling = np.convolve(ep_rewards, np.ones(30)/30, mode='valid')
    plt.plot(range(29, len(ep_rewards)), rolling, label='Trung bình 30 episode', linewidth=2)
plt.xlabel('Episode'); plt.ylabel('Total reward')
plt.title('DQN trên CartPole-v1 (max reward = 500)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Tổng kết

1. **RL** giải bài toán "agent học hành động qua thử-sai-thưởng-phạt" — không cần label cho từng bước.
2. **Q-Learning** học bảng $Q[s,a]$ bằng phương trình Bellman + ε-greedy. Tốt cho state space nhỏ rời rạc.
3. **DQN** thay bảng Q bằng mạng nơ-ron — xử lý được state liên tục. Hai trick quan trọng:
   - **Replay buffer**: phá tương quan thời gian giữa các transition.
   - **Target network**: bản sao Q-network cập nhật chậm, giúp target không nhảy lung tung.
4. Sau DQN còn nhiều thuật toán nâng cao: Double DQN, Dueling DQN, Policy Gradient, A2C, PPO, SAC... đó là các bước tiếp theo nếu em muốn đi sâu vào RL.

# BÀI TẬP VỀ NHÀ

## Bài 1: FrozenLake với is_slippery=True

Train Q-Learning trên `FrozenLake-v1` với `is_slippery=True` (mỗi action có 1/3 xác suất đi đúng hướng, 2/3 trượt sang bên). Bài khó hơn nhiều.

**Yêu cầu:**
1. Chạy với 5000 episode (vì khó hơn cần nhiều thử-sai hơn).
2. Vẽ rolling success rate.
3. So sánh với phiên bản deterministic ở lab — agent đạt tỷ lệ thắng cao nhất là bao nhiêu?
4. Thử thay đổi `alpha` (0.1, 0.5, 0.9), `gamma` (0.9, 0.99) — báo cáo ảnh hưởng.

*Gợi ý:* với is_slippery=True, agent không thể luôn đến đích bằng 1 đường thẳng — phải học chính sách "an toàn" hơn (tránh ô gần hố).

## Bài 2: Cải thiện DQN trên CartPole

Bài lab DQN có thể chưa hội tụ thật tốt trong 300 episode. Hãy:
1. Train 1000 episode. Đến khi nào agent đạt reward 500 (tối đa) đều đặn?
2. Thử các kỹ thuật nâng cao:
   - **Double DQN**: thay `target_net(next_states).max(1)[0]` bằng `target_net(next_states).gather(1, policy_net(next_states).argmax(1, keepdim=True)).squeeze()`. Giảm overestimation bias của Q.
   - **Soft update**: thay vì copy hard mỗi 10 episode, dùng `target_param.data.copy_(τ * policy_param.data + (1-τ) * target_param.data)` với τ = 0.005 mỗi step.
3. Báo cáo: hai kỹ thuật này có giúp hội tụ nhanh hơn không?

## Bài 3: Render và quay video (nâng cao)

Sau khi train xong, render lại một episode bằng tay đã học và lưu thành file MP4.

```python
env_render = gym.make('CartPole-v1', render_mode='rgb_array')
from gymnasium.wrappers import RecordVideo
env_render = RecordVideo(env_render, video_folder='./videos', episode_trigger=lambda x: True)
state, _ = env_render.reset()
done = False
while not done:
    action = select_action(state, eps=0)   # greedy
    state, _, term, trunc, _ = env_render.step(action)
    done = term or trunc
env_render.close()
```

Cần `pip install moviepy`. Nộp video kết quả.

## Bài 4: Đọc thêm

Đọc paper *"Playing Atari with Deep Reinforcement Learning"* (DeepMind, 2013) — bản gốc DQN. Trả lời ngắn:
1. Vì sao DeepMind chọn Atari để demo (chứ không phải robot thật)?
2. Ý nghĩa của **frame stacking** (nối 4 frame liên tiếp làm input)?
3. Kiến trúc mạng dùng cho ảnh Atari là gì? (gợi ý: liên quan tới CNN)

## Hạn nộp
30/04/2026.

# 🎯 BÀI LÀM — Giải bài tập về nhà

> Các cell dưới đây là lời giải cho 4 bài. Mỗi bài viết **độc lập** (tự import thư viện), nên có thể chạy lần lượt từ trên xuống. Nhớ chạy ô cài đặt `pip install gymnasium` ở phần lab trước, và **bật GPU** cho Bài 2 (*Runtime → Change runtime type → GPU*).

---

## ✅ Giải Bài 1: FrozenLake với `is_slippery=True`

Đóng gói Q-Learning thành hàm `train_qlearning(...)` để dùng lại cho mọi cấu hình. Vì môi trường trượt khó hơn nhiều, ta giảm `epsilon` chậm hơn (`eps_decay=0.999`) để agent khám phá lâu hơn trong 5000 episode.

In [ ]:
# Hàm Q-Learning gói gọn để tái sử dụng (cho cả slippery/deterministic và quét hyperparam).
import numpy as np, random
import matplotlib.pyplot as plt

try:
    import gymnasium as gym
except ImportError:
    import gym

def train_qlearning(is_slippery=True, n_episodes=5000, alpha=0.8, gamma=0.95,
                    epsilon=1.0, eps_decay=0.999, eps_min=0.01, seed=42):
    """Trả về (Q_table, rewards_history)."""
    np.random.seed(seed); random.seed(seed)
    env = gym.make('FrozenLake-v1', is_slippery=is_slippery)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    Q = np.zeros((n_states, n_actions))
    eps = epsilon
    rewards_history = []

    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False; total = 0
        while not done:
            if random.random() < eps:               # ε-greedy
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))
            next_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            Q[state, action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])
            state = next_state
            total += reward
        eps = max(eps_min, eps * eps_decay)
        rewards_history.append(total)
    env.close()
    return Q, np.array(rewards_history)

# Yêu cầu 1: train 5000 episode với is_slippery=True.
# Lưu ý: slippery khó hơn -> dùng eps_decay chậm hơn (0.999) để khám phá lâu hơn.
Q_slip, hist_slip = train_qlearning(is_slippery=True, n_episodes=5000,
                                    alpha=0.8, gamma=0.95, eps_decay=0.999)

# Yêu cầu 2: vẽ rolling success rate (cửa sổ 100).
rolling = np.convolve(hist_slip, np.ones(100) / 100, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(rolling)
plt.xlabel('Episode'); plt.ylabel('Tỷ lệ thắng (rolling 100)')
plt.title('Q-Learning FrozenLake is_slippery=True (5000 episode)')
plt.grid(alpha=0.3); plt.show()

print('Tỷ lệ thắng trung bình 500 episode cuối:', round(float(hist_slip[-500:].mean()), 3))

In [ ]:
# Đánh giá chính sách greedy đã học và so sánh slippery vs deterministic.
def eval_greedy(Q, is_slippery=True, n=1000, seed=0):
    env = gym.make('FrozenLake-v1', is_slippery=is_slippery)
    wins = 0
    for i in range(n):
        state, _ = env.reset(seed=seed + i)
        done = False; reward = 0
        while not done:
            action = int(np.argmax(Q[state]))
            state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
        wins += int(reward > 0)
    env.close()
    return wins / n

wr_slip = eval_greedy(Q_slip, is_slippery=True, n=1000)
print(f'Slippery=True   — greedy win-rate: {wr_slip:.3f}')

# Train nhanh bản deterministic để so sánh (như ở lab).
Q_det, _ = train_qlearning(is_slippery=False, n_episodes=2000,
                           alpha=0.8, gamma=0.95, eps_decay=0.995)
wr_det = eval_greedy(Q_det, is_slippery=False, n=1000)
print(f'Slippery=False  — greedy win-rate: {wr_det:.3f}')
print(f'\n=> Trượt (slippery) khó hơn hẳn: trần tỷ lệ thắng thấp hơn nhiều do môi trường ngẫu nhiên.')

In [ ]:
# Yêu cầu 4: quét alpha ∈ {0.1, 0.5, 0.9} và gamma ∈ {0.9, 0.99} (is_slippery=True).
import itertools

alphas = [0.1, 0.5, 0.9]
gammas = [0.9, 0.99]
results = {}

print(f"{'alpha':>6} {'gamma':>6} {'win-rate':>9}")
for a, g in itertools.product(alphas, gammas):
    Q, _ = train_qlearning(is_slippery=True, n_episodes=5000,
                           alpha=a, gamma=g, eps_decay=0.999, seed=0)
    wr = eval_greedy(Q, is_slippery=True, n=1000)
    results[(a, g)] = wr
    print(f'{a:>6} {g:>6} {wr:>9.3f}')

# Heatmap trực quan
mat = np.array([[results[(a, g)] for g in gammas] for a in alphas])
plt.figure(figsize=(5, 4))
plt.imshow(mat, cmap='viridis', aspect='auto', vmin=0, vmax=1)
plt.colorbar(label='greedy win-rate')
plt.xticks(range(len(gammas)), gammas); plt.yticks(range(len(alphas)), alphas)
plt.xlabel('gamma'); plt.ylabel('alpha')
for i in range(len(alphas)):
    for j in range(len(gammas)):
        plt.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center',
                 color='white', fontweight='bold')
plt.title('Win-rate theo alpha × gamma (slippery)')
plt.tight_layout(); plt.show()

### 📝 Nhận xét & báo cáo Bài 1

**3. So sánh với phiên bản deterministic:**
- Bản **deterministic** (`is_slippery=False`): agent học được đường đi thẳng tới đích, tỷ lệ thắng greedy ≈ **100%**.
- Bản **slippery** (`is_slippery=True`): mỗi action chỉ đi đúng hướng với xác suất 1/3, 2/3 trượt sang 2 bên vuông góc → kể cả chính sách **tối ưu** cũng chỉ thắng khoảng **~0.7–0.8** (trên map mặc định, optimal ≈ 0.74). Đây là giới hạn do *bản chất ngẫu nhiên của môi trường*, không phải do agent học kém.

**4. Ảnh hưởng của `alpha` và `gamma`** (xem heatmap ở trên):
- **`alpha` (learning rate):** quá lớn (0.9) → cập nhật giật mạnh theo từng reward ngẫu nhiên, Q-table dao động, khó hội tụ ổn định trong môi trường nhiễu. `alpha` nhỏ/vừa (0.1–0.5) thường cho kết quả **ổn định hơn** với is_slippery=True.
- **`gamma` (discount):** `gamma` cao (0.99) coi trọng reward tương lai → khuyến khích agent kiên nhẫn đi đường vòng *an toàn* tới đích, thường **tốt hơn** `gamma=0.9` ở môi trường mà phần thưởng chỉ đến ở cuối (chỉ có reward=1 tại Goal).
- **Kết luận điển hình:** cấu hình `alpha` thấp + `gamma` cao (vd `alpha≈0.1`, `gamma=0.99`) cho win-rate cao và ổn định nhất. (Hãy điền số liệu cụ thể từ heatmap bạn chạy được vào báo cáo.)

**Gợi ý "chính sách an toàn":** với slippery, agent học cách tránh đứng sát ô hố (H) — vì lỡ trượt là rơi xuống hố — nên đôi khi chọn đường *xa hơn nhưng ít rủi ro hơn*.

## ✅ Giải Bài 2: Cải thiện DQN trên CartPole (Double DQN + Soft update)

Gói toàn bộ vòng lặp DQN vào hàm `train_dqn(...)` với 2 công tắc:
- `double_dqn=True`: dùng **Double DQN** — `policy_net` chọn action, `target_net` đánh giá → giảm overestimation bias.
- `soft_update=True`: cập nhật target network **mượt** mỗi step bằng `τ·policy + (1−τ)·target` (τ=0.005), thay cho copy cứng mỗi 10 episode.

Sau đó huấn luyện **1000 episode** cho 2 cấu hình (cơ bản vs cải tiến), ghi lại episode đầu tiên đạt `avg100 ≥ 475` (mốc "giải được" CartPole-v1) rồi vẽ đồ thị so sánh.

In [ ]:
# Hàm train_dqn gói gọn, hỗ trợ bật/tắt Double DQN và Soft update để so sánh.
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np, random
from collections import deque

try:
    import gymnasium as gym
except ImportError:
    import gym

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Thiết bị:', device)

class QNet(nn.Module):
    def __init__(self, s, a, h=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, a),
        )
    def forward(self, x):
        return self.net(x)

def train_dqn(double_dqn=True, soft_update=True, tau=0.005,
              n_episodes=1000, hard_update_freq=10, target_reward=475, seed=0):
    """Trả về (ep_rewards, solved_at, policy_net)."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    env = gym.make('CartPole-v1')
    sdim = env.observation_space.shape[0]
    adim = env.action_space.n

    policy = QNet(sdim, adim).to(device)
    target = QNet(sdim, adim).to(device)
    target.load_state_dict(policy.state_dict()); target.eval()
    opt = optim.Adam(policy.parameters(), lr=1e-3)
    lossf = nn.MSELoss()

    replay = deque(maxlen=10000)
    batch_size, gamma = 64, 0.99
    eps, eps_decay, eps_min = 1.0, 0.995, 0.05

    ep_rewards = []
    solved_at = None

    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed * 10000 + ep)
        done = False; total = 0
        while not done:
            # ε-greedy
            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    s = torch.FloatTensor(state).unsqueeze(0).to(device)
                    action = int(policy(s).argmax(1).item())

            ns, r, term, trunc, _ = env.step(action)
            done = term or trunc
            replay.append((state, action, r, ns, float(done)))
            state = ns; total += r

            # ---- 1 bước huấn luyện ----
            if len(replay) >= batch_size:
                batch = random.sample(replay, batch_size)
                ss, aa, rr, nss, dd = zip(*batch)
                ss  = torch.FloatTensor(np.array(ss)).to(device)
                aa  = torch.LongTensor(aa).unsqueeze(1).to(device)
                rr  = torch.FloatTensor(rr).to(device)
                nss = torch.FloatTensor(np.array(nss)).to(device)
                dd  = torch.FloatTensor(dd).to(device)

                q_pred = policy(ss).gather(1, aa).squeeze(1)
                with torch.no_grad():
                    if double_dqn:
                        # policy chọn action, target đánh giá -> giảm overestimation
                        next_a = policy(nss).argmax(1, keepdim=True)
                        q_next = target(nss).gather(1, next_a).squeeze(1)
                    else:
                        q_next = target(nss).max(1)[0]
                    q_target = rr + gamma * q_next * (1 - dd)

                loss = lossf(q_pred, q_target)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 10.0)
                opt.step()

                # Soft update: cập nhật target sau mỗi step
                if soft_update:
                    for tp, pp in zip(target.parameters(), policy.parameters()):
                        tp.data.copy_(tau * pp.data + (1 - tau) * tp.data)

        eps = max(eps_min, eps * eps_decay)

        # Hard update: copy nguyên trọng số mỗi vài episode
        if (not soft_update) and ep % hard_update_freq == 0:
            target.load_state_dict(policy.state_dict())

        ep_rewards.append(total)
        avg100 = np.mean(ep_rewards[-100:])
        if solved_at is None and len(ep_rewards) >= 100 and avg100 >= target_reward:
            solved_at = ep + 1
            print(f'  >> Đạt avg100 ≥ {target_reward} tại episode {solved_at}')
        if (ep + 1) % 50 == 0:
            print(f'  Ep {ep+1:4d}  avg100 = {avg100:6.1f}  eps = {eps:.3f}')

    env.close()
    return ep_rewards, solved_at, policy

In [ ]:
# Huấn luyện 1000 episode cho mỗi phiên bản và so sánh.
# (Nặng — nên bật GPU. Muốn nhanh khi thử: đổi n_episodes=500.)
print('=== (A) DQN CƠ BẢN: hard update mỗi 10 ep, KHÔNG Double ===')
rew_base, solved_base, net_base = train_dqn(double_dqn=False, soft_update=False,
                                            n_episodes=1000, seed=0)

print('\n=== (B) DQN CẢI TIẾN: Double DQN + Soft update (τ=0.005) ===')
rew_imp, solved_imp, net_imp = train_dqn(double_dqn=True, soft_update=True,
                                         tau=0.005, n_episodes=1000, seed=0)

print(f'\n>>> Episode giải xong (avg100 ≥ 475):  base = {solved_base},  improved = {solved_imp}')

In [ ]:
# So sánh đường học của 2 phiên bản.
import matplotlib.pyplot as plt

def smooth(x, w=30):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(11, 4))
plt.plot(smooth(rew_base), label=f'DQN cơ bản (solved @ {solved_base})')
plt.plot(smooth(rew_imp),  label=f'Double DQN + Soft update (solved @ {solved_imp})')
plt.axhline(475, color='gray', ls='--', alpha=0.6, label='ngưỡng giải (475)')
plt.axhline(500, color='green', ls=':', alpha=0.4, label='reward tối đa (500)')
plt.xlabel('Episode'); plt.ylabel('Reward (trung bình 30 episode)')
plt.title('So sánh DQN cơ bản vs Double DQN + Soft update (CartPole, 1000 ep)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 📝 Báo cáo Bài 2

So sánh dựa trên biến `solved_base` và `solved_imp` (episode đầu tiên mà reward trung bình 100 episode gần nhất ≥ 475) cùng đồ thị phía trên:

- **Double DQN** tách việc *chọn* action (dùng `policy_net`) khỏi việc *đánh giá* action (dùng `target_net`), nên giảm **overestimation bias** — DQN cơ bản thường *thổi phồng* (phóng đại) ước lượng Q vì toán tử `max`. Nhờ vậy đường học mượt hơn, ít tụt dốc đột ngột.
- **Soft update** (`τ=0.005` mỗi step) làm target network bám theo policy network một cách *từ từ, liên tục* thay vì copy cứng mỗi 10 episode. Target ổn định hơn → training ít dao động.
- **Kết luận điển hình:** bản Double DQN + Soft update thường **đạt mốc 475 sớm hơn** và **giữ ổn định ở mức 500 đều đặn hơn** so với DQN cơ bản. (Kết quả cụ thể có thể dao động theo seed/phần cứng — hãy điền số `solved_base` vs `solved_imp` in ra ở trên vào báo cáo của bạn.)

> Ghi chú: huấn luyện 2 × 1000 episode khá nặng trên CPU (~10–20 phút). Nên bật **GPU** trong Colab (*Runtime → Change runtime type → GPU*), hoặc giảm `n_episodes` xuống 500 để chạy nhanh hơn khi thử.

## ✅ Giải Bài 3: Render & quay video CartPole

Dùng lại policy đã huấn luyện ở Bài 2 (`net_imp`), chạy 1 episode greedy với `render_mode='rgb_array'` và bọc môi trường bằng `RecordVideo` để xuất MP4. Cell cuối hiển thị video trực tiếp trong Colab.

> ⚠️ Cần chạy Bài 2 trước (để có biến `net_imp` và `device`).

In [ ]:
# Render lại 1 episode bằng policy đã học (net_imp từ Bài 2) và lưu MP4.
# Cần moviepy để gymnasium ghi video.
!pip install -q moviepy

import os, glob
import torch
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

os.makedirs('videos', exist_ok=True)
env_render = gym.make('CartPole-v1', render_mode='rgb_array')
env_render = RecordVideo(env_render, video_folder='./videos',
                         episode_trigger=lambda x: True, name_prefix='cartpole')

state, _ = env_render.reset(seed=123)
done = False
total = 0
while not done:
    with torch.no_grad():                       # greedy, không exploration
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        action = int(net_imp(s).argmax(1).item())
    state, r, term, trunc, _ = env_render.step(action)
    done = term or trunc
    total += r
env_render.close()
print(f'Tổng reward khi render (greedy): {total}')
print('Video đã lưu trong ./videos')

In [ ]:
# Hiển thị video ngay trong Colab.
from IPython.display import HTML
from base64 import b64encode
import glob

mp4 = sorted(glob.glob('videos/*.mp4'))[-1]
data = b64encode(open(mp4, 'rb').read()).decode()
HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## ✅ Giải Bài 4: Đọc paper "Playing Atari with Deep Reinforcement Learning" (DeepMind, 2013)

**1. Vì sao chọn Atari để demo (thay vì robot thật)?**
- Atari là một **benchmark chuẩn hoá**: 49 game đa dạng, cùng một dạng input (ảnh màn hình) và output (joystick), nên có thể dùng **đúng một kiến trúc + một bộ hyperparameter** cho mọi game → chứng minh được tính **tổng quát** của thuật toán.
- Môi trường mô phỏng chạy **nhanh, rẻ, an toàn**: agent có thể chơi hàng triệu frame, reset tức thì. Robot thật thì chậm, đắt, dễ hư hỏng, nguy hiểm và rất khó lặp lại thí nghiệm.
- **Reward có sẵn** dưới dạng điểm số của game — không cần con người gán nhãn hay thiết kế reward phức tạp.

**2. Ý nghĩa của frame stacking (nối 4 frame liên tiếp làm input)?**
- Một ảnh tĩnh **không chứa thông tin chuyển động**: nhìn 1 frame không biết quả bóng đang bay sang trái hay sang phải, nhanh hay chậm.
- Nối 4 frame liên tiếp cho phép mạng **suy ra vận tốc và hướng di chuyển** của các vật thể → state trở nên gần với tính chất **Markov** hơn (đủ thông tin để ra quyết định tối ưu mà không cần nhớ quá khứ xa).

**3. Kiến trúc mạng cho ảnh Atari?**
- Là một **CNN (Convolutional Neural Network)**. Input là ảnh xám 84×84×4 (4 frame stack). Cấu trúc gốc:
  - Conv1: 16 filter 8×8, stride 4, ReLU
  - Conv2: 32 filter 4×4, stride 2, ReLU
  - Fully-connected 256 units, ReLU
  - Output: 1 neuron cho mỗi action (giá trị Q của từng action) — chỉ cần 1 lần forward để lấy Q cho tất cả action.
- (Bản Nature 2015 sau đó mở rộng thành 3 lớp conv: 32×8×8/4, 64×4×4/2, 64×3×3/1.)